# S13v5b — SEVIR GraphConvLSTM v5b — Kaggle

**v5'ten tek fark — Balanced MSE (B-MSE) loss:**
- Plain `nn.MSELoss()` → `bmse_loss` ile değiştirildi.
- Pixel-bazlı kademeli ağırlık (SEVIR/PreDiff CSI threshold'larıyla aligned):
  - `< 16/255` (arka plan) → w=1
  - `16-74/255` → w=2
  - `74-133/255` → w=5
  - `133-160/255` → w=10
  - `160-181/255` → w=15
  - `181-219/255` → w=20
  - `>= 219/255` → w=30
- Çarpanlar Shi 2017 (HKO-7) inspired. İlk denemede stabilite > agresiflik.

**Hedef:** heavy precip CSI'da kazanç (v5 test: thr=219 için 0.0927; thr=181 için 0.1961). Plain MSE'de gradient'in büyük kısmı arka plandan geliyor → mean prediction (blur). B-MSE bu basıncı dengeliyor.

**WMSE α=10'dan farkı:** tek-cliff binary değil, 6 kademeli. Light precip'i kaybetmeden heavy precip'e basınç verir.

**Mimari değişiklik YOK:** v5 ile birebir (sigmoid kaldırılmış, GroupNorm, dinamik Gaussian edge_attr, residual `h_next + x`, GAT 2-layer/4-head, FEAT_DIM=HIDDEN_DIM=64).

**Eğitim hyperparam:** v5 ile aynı (lr=1e-3, batch=4, EPOCHS=100, PATIENCE=15, AMP). Hem train hem val B-MSE kullanır; `evaluate_metrics` plain MSE/RMSE/SSIM/CSI raporlar.

**Kabul kriteri:**
- test CSI-M v5'i (0.3823) geçmeli — hedef ≥ 0.39
- thr=181 ve thr=219 CSI'sı en az 1.5× v5 değeri olmalı
- MSE/RMSE biraz kötüleşebilir (sharpness ↔ L2 trade) — beklenen

**Başlamadan önce:**
1. Kaggle → Datasets → `sevir3` ekle
2. Accelerator → **GPU T4 x2**
3. Settings → Internet → **On**
4. **Save & Run All (Commit)**

**Çıktılar:** `sevir_graphconvlstm_v5b.pt`, `sevir_metrics_graphconvlstm_v5b.json`, `training_history.png`, `sevir_graphconvlstm_v5b_comparison.gif`

In [ ]:
import subprocess, sys, torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

try:
    import torch_geometric
    print(f'OK - torch_geometric mevcut: {torch_geometric.__version__}')
except ImportError:
    print('torch_geometric YOK - kurulum deneniyor (PyPI explicit)...')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--index-url', 'https://pypi.org/simple/',
         'torch_geometric'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('STDOUT:', r.stdout[-1500:])
        print('STDERR:', r.stderr[-1500:])
        raise RuntimeError(
            'torch_geometric kurulamadi. Settings -> Internet ON olmali; '
            'pip ciktisindan asil sebebi gor.'
        )
    import torch_geometric
    print(f'OK - kuruldu: {torch_geometric.__version__}')


In [ ]:
import os, json, time, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as col
import imageio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.nn import GATConv
from skimage.metrics import structural_similarity as sk_ssim

INPUT_DIR  = '/kaggle/input/datasets/vartoy/sevir3'
OUTPUT_DIR = '/kaggle/working'

TRAIN_FILE   = os.path.join(INPUT_DIR,  'sevir_vil_train_subset.npy')
VAL_FILE     = os.path.join(INPUT_DIR,  'sevir_vil_val_subset.npy')
TEST_FILE    = os.path.join(INPUT_DIR,  'sevir_vil_test_subset.npy')
MODEL_PATH   = os.path.join(OUTPUT_DIR, 'sevir_graphconvlstm_v5b.pt')
METRICS_PATH = os.path.join(OUTPUT_DIR, 'sevir_metrics_graphconvlstm_v5b.json')
FIG_DIR      = OUTPUT_DIR

N_INPUT   = 7     # 7 girdi kare (10dk aralık, PreDiff protokolü)
N_PREDICT = 6     # 6 tahmin kare
GRID_SIZE = 32
FEAT_DIM  = 64
HIDDEN_DIM = 64
GAT_HEADS  = 4
N_LAYERS   = 2
SIGMA      = 1.0

EPOCHS     = 100
BATCH_SIZE = 4
PATIENCE   = 15
LR         = 1e-3

CSI_THRESHOLDS = [16/255, 74/255, 133/255, 160/255, 181/255, 219/255]

# B-MSE çarpanları — bin sınırları CSI_THRESHOLDS ile aynı.
# Index karşılığı: [<16, 16-74, 74-133, 133-160, 160-181, 181-219, >=219]
BMSE_WEIGHTS = [1.0, 2.0, 5.0, 10.0, 15.0, 20.0, 30.0]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

HAS_TEST = os.path.exists(TEST_FILE)
for f in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1e9
        print(f'{os.path.basename(f)}: {size:.2f} GB')
    else:
        print(f'YOK: {os.path.basename(f)}  (test ise atlanır)')

print(f'BMSE_WEIGHTS: {BMSE_WEIGHTS}')

In [ ]:
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

g = torch.Generator()
g.manual_seed(42)

def _seed_worker(worker_id):
    ws = 42 + worker_id
    np.random.seed(ws)
    random.seed(ws)


class SEVIRDataset(torch.utils.data.Dataset):
    def __init__(self, filepath):
        print(f'Yükleniyor (memmap): {filepath}')
        self.data = np.load(filepath, mmap_mode='r')
        self.n = self.data.shape[0]
        print(f'  {self.n} örnek  {self.data.shape}')

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        seq = self.data[idx].astype(np.float32)
        x = torch.from_numpy(seq[:N_INPUT]).unsqueeze(1)
        y = torch.from_numpy(seq[N_INPUT:]).unsqueeze(1)
        return x, y

train_ds = SEVIRDataset(TRAIN_FILE)
val_ds   = SEVIRDataset(VAL_FILE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True,
                          worker_init_fn=_seed_worker, generator=g)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True,
                          worker_init_fn=_seed_worker, generator=g)

if HAS_TEST:
    test_ds = SEVIRDataset(TEST_FILE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True,
                             worker_init_fn=_seed_worker, generator=g)
else:
    test_ds = test_loader = None
    print('Test seti bulunamadı — test değerlendirmesi atlanacak.')

print('DataLoader hazır.')


In [ ]:
# Graph topolojisi (statik) — ağırlıklar dinamik hesaplanacak

def build_grid_edges(H=GRID_SIZE, W=GRID_SIZE):
    edges = []
    for i in range(H):
        for j in range(W):
            node = i * W + j
            for di in [-1, 0, 1]:
                for dj in [-1, 0, 1]:
                    if di == 0 and dj == 0:
                        continue
                    ni, nj = i + di, j + dj
                    if 0 <= ni < H and 0 <= nj < W:
                        edges.append([node, ni * W + nj])
    return torch.tensor(edges, dtype=torch.long).t().contiguous()

def batch_edge_index(edge_index_single, batch_size, num_nodes):
    return torch.cat([edge_index_single + i * num_nodes
                      for i in range(batch_size)], dim=1)

def compute_edge_attr(x_nodes, edge_index, sigma=SIGMA):
    """
    w = exp(-||h_i - h_j||² / 2σ²)
    Her adımda encoder çıktısından hesaplanır → dinamik.
    """
    src, dst = edge_index[0], edge_index[1]
    diff    = x_nodes[src] - x_nodes[dst]             # (E, feat_dim)
    dist_sq = (diff ** 2).sum(dim=-1, keepdim=True)   # (E, 1)
    return torch.exp(-dist_sq / (2 * sigma ** 2))      # (E, 1)

In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self, in_channels=1, out_channels=FEAT_DIM):
        super().__init__()
        mid = out_channels // 2  # 32
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid, 3, padding=1),
            nn.GroupNorm(8, mid), nn.LeakyReLU(0.2),
            nn.Conv2d(mid, mid, 3, padding=1),
            nn.GroupNorm(8, mid), nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),
            nn.Conv2d(mid, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.LeakyReLU(0.2),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.net(x)


class CNNDecoder(nn.Module):
    def __init__(self, in_channels=HIDDEN_DIM + FEAT_DIM, out_channels=1):
        super().__init__()
        mid = HIDDEN_DIM // 2  # 32
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_channels, mid, 2, stride=2),
            nn.GroupNorm(8, mid), nn.LeakyReLU(0.2),
            nn.Conv2d(mid, mid, 3, padding=1),
            nn.GroupNorm(8, mid), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(mid, mid // 2, 2, stride=2),
            nn.GroupNorm(4, mid // 2), nn.LeakyReLU(0.2),
            nn.Conv2d(mid // 2, out_channels, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
class GraphConvLSTMCell(nn.Module):
    def __init__(self, in_feat, hidden_dim=HIDDEN_DIM, heads=GAT_HEADS):
        super().__init__()
        assert in_feat == hidden_dim, (
            f"Residual connection h_next + x requires in_feat == hidden_dim, "
            f"got in_feat={in_feat}, hidden_dim={hidden_dim}"
        )
        self.hidden_dim = hidden_dim
        self.gat = GATConv(
            in_channels  = in_feat + hidden_dim,
            out_channels = (4 * hidden_dim) // heads,
            heads        = heads,
            concat       = True,
            edge_dim     = 1
        )

    def forward(self, x, h, c, edge_index, edge_attr):
        gates = self.gat(torch.cat([x, h], dim=-1), edge_index,
                         edge_attr=edge_attr)
        i, f, g, o = gates.chunk(4, dim=-1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        h_next = h_next + x   # residual
        return h_next, c_next


class SEVIRGraphConvLSTMv5b(nn.Module):
    """
    v5b: v5 ile birebir aynı mimari. Tek fark eğitimde — B-MSE loss.
    """
    def __init__(self, in_channels=1, feat_dim=FEAT_DIM,
                 hidden_dim=HIDDEN_DIM, heads=GAT_HEADS,
                 n_layers=N_LAYERS, grid_size=GRID_SIZE):
        super().__init__()
        self.grid_size  = grid_size
        self.num_nodes  = grid_size * grid_size
        self.feat_dim   = feat_dim
        self.hidden_dim = hidden_dim
        self.n_layers   = n_layers

        self.encoder = CNNEncoder(in_channels, feat_dim)
        self.decoder = CNNDecoder(hidden_dim + feat_dim, in_channels)

        self.cells = nn.ModuleList([
            GraphConvLSTMCell(
                in_feat    = feat_dim if i == 0 else hidden_dim,
                hidden_dim = hidden_dim,
                heads      = heads
            )
            for i in range(n_layers)
        ])

        self.register_buffer('edge_index_single',
                             build_grid_edges(grid_size, grid_size))

    def _step(self, frame, h_list, c_list, B, edge_index):
        N    = self.num_nodes
        feat = self.encoder(frame)
        x    = feat.flatten(2).permute(0, 2, 1).reshape(B * N, self.feat_dim)

        # Dinamik Gaussian edge_attr
        edge_attr = compute_edge_attr(x, edge_index)

        new_h, new_c = [], []
        for i, cell in enumerate(self.cells):
            h, c = cell(x, h_list[i], c_list[i], edge_index, edge_attr)
            new_h.append(h)
            new_c.append(c)
            x = h
        return new_h, new_c, feat

    def _decode(self, h_last, enc_feat, B):
        # v5/v5b: sigmoid kaldırıldı — raw output. Inference'ta np.clip(0,1) yapılıyor.
        N    = self.num_nodes
        fmap = h_last.reshape(B, N, self.hidden_dim) \
                     .permute(0, 2, 1) \
                     .reshape(B, self.hidden_dim, self.grid_size, self.grid_size)
        return self.decoder(torch.cat([fmap, enc_feat], dim=1))

    def forward(self, x_in):
        B, T, C, H, W = x_in.shape
        N = self.num_nodes

        edge_index = batch_edge_index(
            self.edge_index_single, B, N).to(x_in.device)

        h_list = [torch.zeros(B * N, self.hidden_dim, device=x_in.device)
                  for _ in range(self.n_layers)]
        c_list = [torch.zeros(B * N, self.hidden_dim, device=x_in.device)
                  for _ in range(self.n_layers)]

        enc_feat = None
        for t in range(T):
            h_list, c_list, enc_feat = self._step(
                x_in[:, t], h_list, c_list, B, edge_index)

        pred_frame = x_in[:, -1]
        outputs = []
        for _ in range(N_PREDICT):
            h_list, c_list, enc_feat = self._step(
                pred_frame, h_list, c_list, B, edge_index)
            pred_raw = self._decode(h_list[-1], enc_feat, B)
            outputs.append(pred_raw.unsqueeze(1))
            # Autoregressive feedback: encoder valid range [0,1] görmeli.
            # Output raw kalıyor — loss gradient'i clamp boundary'sinde de flow ediyor.
            pred_frame = pred_raw.clamp(0, 1)

        return torch.cat(outputs, dim=1)


model = SEVIRGraphConvLSTMv5b().to(DEVICE)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Toplam parametre: {total:,}')
print(f'v5b: v5 mimarisi + B-MSE loss (ağırlıklar: {BMSE_WEIGHTS})')
print(f'Sigma: {SIGMA}')

In [ ]:
def bmse_loss(pred, target):
    """
    Balanced MSE — Shi 2017 inspired. Pixel intensity bin'ine göre kademeli ağırlık.
    Bin sınırları SEVIR/PreDiff CSI threshold'larıyla aynı.
    """
    w = torch.ones_like(target)
    w = torch.where(target >= 16/255,  torch.full_like(w, BMSE_WEIGHTS[1]), w)
    w = torch.where(target >= 74/255,  torch.full_like(w, BMSE_WEIGHTS[2]), w)
    w = torch.where(target >= 133/255, torch.full_like(w, BMSE_WEIGHTS[3]), w)
    w = torch.where(target >= 160/255, torch.full_like(w, BMSE_WEIGHTS[4]), w)
    w = torch.where(target >= 181/255, torch.full_like(w, BMSE_WEIGHTS[5]), w)
    w = torch.where(target >= 219/255, torch.full_like(w, BMSE_WEIGHTS[6]), w)
    return (w * (pred - target) ** 2).mean()


optimizer  = torch.optim.Adam(model.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
                 optimizer, patience=5, factor=0.5, min_lr=1e-6)
criterion  = bmse_loss
scaler     = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
best_val   = float('inf')
no_improve = 0
history    = {'train_loss': [], 'val_loss': [], 'train_rmse': [], 'val_rmse': []}

train_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            pred = model(xb)
            loss = criterion(pred, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                val_loss += criterion(model(xb), yb).item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_rmse'].append(float(np.sqrt(train_loss)))
    history['val_rmse'].append(float(np.sqrt(val_loss)))
    scheduler.step(val_loss)

    elapsed       = time.time() - epoch_start
    total_elapsed = time.time() - train_start
    print(f'Epoch {epoch:3d}/{EPOCHS}  '
          f'Train BMSE: {train_loss:.5f}  sqrt: {np.sqrt(train_loss):.4f}  |  '
          f'Val BMSE: {val_loss:.5f}  sqrt: {np.sqrt(val_loss):.4f}  '
          f'[{elapsed:.0f}s / toplam {total_elapsed/60:.1f}dk]')

    if val_loss < best_val:
        best_val   = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'EarlyStopping tetiklendi ({PATIENCE} epoch).')
            break

toplam = (time.time() - train_start) / 60
print(f'\nEn iyi Val BMSE: {best_val:.5f}  |  Toplam süre: {toplam:.1f} dk')
print(f'Not: train/val "loss" B-MSE skalasında; plain MSE/RMSE evaluate_metrics çıktısında.')

In [ ]:
def compute_csi(pred_np, true_np, threshold):
    p, t = pred_np >= threshold, true_np >= threshold
    tp, fp, fn = (p & t).sum(), (p & ~t).sum(), (~p & t).sum()
    d = tp + fp + fn
    return float(tp / d) if d > 0 else 0.0


def _max_pool_np(arr, pool_size):
    # PreDiff/DGMR convention: max-pool then binarize for CSI-pool metrics
    t = torch.from_numpy(arr).float().unsqueeze(1)
    pooled = F.max_pool2d(t, kernel_size=pool_size, stride=pool_size)
    return pooled.squeeze(1).numpy()


def compute_csi_pool(pred_flat, true_flat, threshold, pool_size):
    pp = _max_pool_np(pred_flat, pool_size)
    tp = _max_pool_np(true_flat, pool_size)
    return compute_csi(pp, tp, threshold)


def evaluate_metrics(model, loader):
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for xb, yb in loader:
            pred = model(xb.to(DEVICE)).cpu().numpy()
            pred = np.clip(pred, 0.0, 1.0)   # v5/v5b: sigmoid yok -> output clip
            all_pred.append(pred[:, :, 0])
            all_true.append(yb.numpy()[:, :, 0])
    pred_all = np.concatenate(all_pred)
    true_all = np.concatenate(all_true)
    N, T_pred, H, W = pred_all.shape

    ssim_matrix = np.zeros((N, T_pred), dtype=np.float64)
    for i in range(N):
        for t in range(T_pred):
            ssim_matrix[i, t] = sk_ssim(
                pred_all[i, t], true_all[i, t], data_range=1.0)

    per_lt_mse, per_lt_ssim, per_lt_csi_m, per_lt_csi_per_thr = [], [], [], []
    for t in range(T_pred):
        p_t = pred_all[:, t]
        y_t = true_all[:, t]
        mse_t = float(np.mean((p_t - y_t) ** 2))
        per_lt_mse.append(mse_t)
        per_lt_ssim.append(float(ssim_matrix[:, t].mean()))
        csi_t = [compute_csi(p_t, y_t, thr) for thr in CSI_THRESHOLDS]
        per_lt_csi_per_thr.append([round(c, 4) for c in csi_t])
        per_lt_csi_m.append(float(np.mean(csi_t)))

    pred_flat = pred_all.reshape(-1, H, W)
    true_flat = true_all.reshape(-1, H, W)
    csi_scores = [compute_csi(pred_flat, true_flat, thr) for thr in CSI_THRESHOLDS]
    csi_pool4  = [compute_csi_pool(pred_flat, true_flat, thr, 4)  for thr in CSI_THRESHOLDS]
    csi_pool16 = [compute_csi_pool(pred_flat, true_flat, thr, 16) for thr in CSI_THRESHOLDS]
    mse = float(np.mean((pred_all - true_all) ** 2))

    return {
        'mse': mse,
        'rmse': float(np.sqrt(mse)),
        'ssim': float(ssim_matrix.mean()),
        'csi_m': float(np.mean(csi_scores)),
        'csi_per_threshold': [round(c, 4) for c in csi_scores],
        'csi_pool4_m':  float(np.mean(csi_pool4)),
        'csi_pool4_per_threshold':  [round(c, 4) for c in csi_pool4],
        'csi_pool16_m': float(np.mean(csi_pool16)),
        'csi_pool16_per_threshold': [round(c, 4) for c in csi_pool16],
        'per_leadtime': {
            'mse':  [round(float(m), 6) for m in per_lt_mse],
            'rmse': [round(float(np.sqrt(m)), 6) for m in per_lt_mse],
            'ssim': [round(float(s), 4) for s in per_lt_ssim],
            'csi_m': [round(float(c), 4) for c in per_lt_csi_m],
            'csi_per_threshold': per_lt_csi_per_thr,
        },
    }


model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))

print('Metrikler hesaplaniyor (val)...')
val_metrics = evaluate_metrics(model, val_loader)

if HAS_TEST:
    print('Metrikler hesaplaniyor (test)...')
    test_metrics = evaluate_metrics(model, test_loader)
else:
    test_metrics = None

out = {
    'model':           'SEVIRGraphConvLSTMv5b',
    'loss':            'B-MSE',
    'bmse_weights':    BMSE_WEIGHTS,
    'epochs_ran':      len(history['train_loss']),
    'best_val_bmse':   round(min(history['val_loss']), 6),
    'val':  val_metrics,
    'test': test_metrics,
}

with open(METRICS_PATH, 'w') as f:
    json.dump(out, f, indent=2)

print(f"\n--- SEVIR GraphConvLSTM v5b Sonuclar ---")
print(f"  Best Val BMSE : {out['best_val_bmse']:.6f}")
print(f"  Epoch         : {out['epochs_ran']}")
print(f"  -- Validation (plain MSE on output) --")
print(f"  MSE        : {val_metrics['mse']:.6f}")
print(f"  RMSE       : {val_metrics['rmse']:.6f}")
print(f"  SSIM       : {val_metrics['ssim']:.4f}")
print(f"  CSI-M      : {val_metrics['csi_m']:.4f}")
print(f"  CSI-pool4  : {val_metrics['csi_pool4_m']:.4f}")
print(f"  CSI-pool16 : {val_metrics['csi_pool16_m']:.4f}")
if test_metrics is not None:
    print(f"  -- Test (plain MSE on output) --")
    print(f"  MSE        : {test_metrics['mse']:.6f}")
    print(f"  RMSE       : {test_metrics['rmse']:.6f}")
    print(f"  SSIM       : {test_metrics['ssim']:.4f}")
    print(f"  CSI-M      : {test_metrics['csi_m']:.4f}  (PreDiff Tab.2 ConvLSTM: 0.4185 | v5: 0.3823)")
    print(f"  CSI-pool4  : {test_metrics['csi_pool4_m']:.4f}  (PreDiff Tab.2 ConvLSTM: 0.4452 | v5: 0.3949)")
    print(f"  CSI-pool16 : {test_metrics['csi_pool16_m']:.4f}  (PreDiff Tab.2 ConvLSTM: 0.5135 | v5: 0.4344)")
    print(f"  CSI@thr=181/255 : {test_metrics['csi_per_threshold'][4]:.4f}  (v5: 0.1961)")
    print(f"  CSI@thr=219/255 : {test_metrics['csi_per_threshold'][5]:.4f}  (v5: 0.0927)")
print(f"\nMetrikler: {METRICS_PATH}")

In [ ]:
colors = ['white', '#8ec7ff', 'dodgerblue', 'lime', 'yellow', 'darkorange', 'red', 'darkmagenta']
CMAP   = col.LinearSegmentedColormap.from_list('radar', colors)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train BMSE')
axes[0].plot(history['val_loss'],   label='Val BMSE')
axes[0].set_title('B-MSE Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history['train_rmse'], label='Train sqrt(BMSE)')
axes[1].plot(history['val_rmse'],   label='Val sqrt(BMSE)')
axes[1].set_title('sqrt(B-MSE)'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'training_history.png'), dpi=150)
plt.show()

sample_x, sample_y = val_ds[0]
model.eval()
with torch.no_grad():
    pred = model(sample_x.unsqueeze(0).to(DEVICE))[0].cpu().numpy()
    pred = np.clip(pred, 0.0, 1.0)   # v5b: sigmoid yok -> sample clip

frames = []
for i in range(N_PREDICT):
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(sample_y[i, 0].numpy(), cmap=CMAP, vmin=0, vmax=1)
    axes[0].set_title(f'Gercek t+{(i+1)*10}dk'); axes[0].axis('off')
    axes[1].imshow(pred[i, 0], cmap=CMAP, vmin=0, vmax=1)
    axes[1].set_title(f'Tahmin t+{(i+1)*10}dk'); axes[1].axis('off')
    plt.tight_layout()
    fig.canvas.draw()
    frames.append(np.asarray(fig.canvas.buffer_rgba())[..., :3])
    plt.close(fig)

gif_path = os.path.join(FIG_DIR, 'sevir_graphconvlstm_v5b_comparison.gif')
imageio.mimsave(gif_path, frames, fps=3)
print(f'GIF: {gif_path}')
print('Tamamlandi.')